# 03. Vectorization & Universal Functions (5+ Years Interview Guide)
Exhaustive revision guide to SIMD C-level vectorization, math ufuncs, and ufunc methods (.reduce, .accumulate, .outer) on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Arithmetic Vectorization**: Element-wise `+`, `-`, `*`, `/`, `**` executing in C without GIL contention.
- **Mathematical ufuncs**: Dedicated cell for `np.sqrt()`, `np.exp()`, `np.log()`, `np.sin()`, and `np.abs()`.
- **Ufunc Reduction Methods**: Dedicated cell for `.reduce()`, `.accumulate()`, and `.outer()`.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [ ]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

### Arithmetic Vectorization (`+`, `-`, `*`, `/`)
**Explanation**: Applies fee multipliers and taxes element-wise to transaction amounts.

**Syntax**: `amounts[:5] * 1.05`

In [ ]:
print('Amounts with 5% Fee Applied (first 5):', (amounts[:5] * 1.05).round(2))

### Square Root: `np.sqrt()`
**Explanation**: Computes square root of transaction amounts for variance scaling.

**Syntax**: `np.sqrt(amounts[:5])`

In [ ]:
print('Square Roots of Amounts:', np.sqrt(amounts[:5]).round(2))

### Exponential: `np.exp()`
**Explanation**: Computes exponential scaling for risk logits.

**Syntax**: `np.exp(account_ages[:5] / 12.0)`

In [ ]:
print('Exponential Account Age Factors:', np.exp(account_ages[:5] / 12.0).round(2))

### Natural Logarithm: `np.log()`
**Explanation**: Applies log-transformation $\ln(1 + x)$ to normalize skewed transaction amounts.

**Syntax**: `np.log1p(amounts[:5])`

In [ ]:
log_amounts = np.log1p(amounts[:5])
print('Log-Transformed Amounts (ln(1+x)):', log_amounts.round(3))

### Trigonometric Functions: `np.sin()` & `np.cos()`
**Explanation**: Computes cyclical trigonometric time encodings.

**Syntax**: `np.sin(2 * np.pi * day_of_year / 365.25)`

In [ ]:
cyclical_feature = np.sin(2 * np.pi * (amounts[:5] % 365) / 365)
print('Cyclical Encodings:', cyclical_feature.round(3))

### Absolute Value: `np.abs()`
**Explanation**: Computes absolute deviation from median transaction amount.

**Syntax**: `np.abs(amounts[:5] - np.median(amounts))`

In [ ]:
abs_devs = np.abs(amounts[:5] - np.median(amounts))
print('Absolute Deviations from Median:', abs_devs.round(2))

### Cumulative Reductions with `ufunc.reduce()`
**Explanation**: Calculates total revenue using `np.add.reduce`.

**Syntax**: `np.add.reduce(amounts)`

In [ ]:
total_revenue = np.add.reduce(amounts)
print(f'Total Transaction Volume (np.add.reduce): ${total_revenue:,.2f}')

### Running Totals with `ufunc.accumulate()`
**Explanation**: Calculates running cumulative revenue stream across transactions.

**Syntax**: `np.add.accumulate(amounts[:5])`

In [ ]:
running_revenue = np.add.accumulate(amounts[:5])
print('Running Cumulative Revenue (first 5):', running_revenue.round(2))

### Outer Products with `ufunc.outer()`
**Explanation**: Computes outer cross-product risk matrix between card fee tiers and account ages.

**Syntax**: `np.multiply.outer(fee_rates, account_ages[:4])`

In [ ]:
fees = np.array([0.015, 0.025, 0.035])
outer_fee_matrix = np.multiply.outer(fees, account_ages[:4])
print('Outer Fee Scaling Matrix (3 fees x 4 accounts):\n', outer_fee_matrix.round(2))

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Pairwise Transaction Distance Matrix via `np.subtract.outer`
**Explanation**: Compute the complete pairwise absolute difference matrix for the first 5 transaction amounts without Python loops.

**Syntax**: `np.abs(np.subtract.outer(amounts[:5], amounts[:5]))`

In [ ]:
pw_matrix = np.abs(np.subtract.outer(amounts[:5], amounts[:5]))
print('Pairwise Amount Differences Matrix:\n', pw_matrix.round(2))